## Import libraries

In [1]:
import pandas as pd
from langchain_community.vectorstores import FAISS

import sys
sys.path.append("..")

from src.embedding import (
    create_stratified_sample,
    split_documents,
    create_embeddings,
    build_faiss_index
)

C:\Users\hp\AppData\Local\Temp\ipykernel_17172\3568671973.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## Load cleaned data

In [2]:
df = pd.read_csv("../data/processed/cleaned_complaints.csv")

df.head()

,Complaint ID,Date received,product_normalized,Product,Sub-product,Issue,Sub-issue,narrative_clean,narrative_word_count_clean,Company,State,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?
0,14069121,2025-06-13,Credit Card,Credit card,Store credit card,Getting a credit card,Card opened without my consent or knowledge,a xxxx xxxx card was opened under my name by a...,91,"CITIBANK, N.A.",TX,Web,2025-06-13,Closed with non-monetary relief,Yes,NaN
1,14061897,2025-06-13,Savings Account,Checking or savings account,Checking account,Managing an account,Deposits and withdrawals,i made the mistake of using my wellsfargo debi...,108,WELLS FARGO & COMPANY,ID,Web,2025-06-13,Closed with explanation,Yes,NaN
2,14047085,2025-06-12,Credit Card,Credit card,General-purpose credit card or charge card,"Other features, terms, or problems",Other problem,dear cfpb i have a secured credit card with ci...,127,"CITIBANK, N.A.",NY,Web,2025-06-13,Closed with monetary relief,Yes,NaN
3,14040217,2025-06-12,Credit Card,Credit card,General-purpose credit card or charge card,Incorrect information on your report,Account information incorrect,i have a citi rewards cards the credit balance...,234,"CITIBANK, N.A.",IL,Web,2025-06-12,Closed with explanation,Yes,NaN
4,13968411,2025-06-09,Credit Card,Credit card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,b'i am writing to dispute the following charge...,503,"CITIBANK, N.A.",TX,Web,2025-06-09,Closed with monetary relief,Yes,NaN


## Stratified Sampling

In [3]:
sample_df = create_stratified_sample(
    df,
    sample_size=12000,
    random_state=42
)

## Display sample size

In [4]:
print(sample_df.shape)

(12000, 16)


## Check proportions

In [5]:
sample_df["product_normalized"].value_counts(normalize=True)

product_normalized
Credit Card        0.361083
Savings Account    0.296000
Money Transfer     0.188250
Personal Loan      0.154667
Name: proportion, dtype: float64

## Compare to original

In [6]:
df["product_normalized"].value_counts(normalize=True)

product_normalized
Credit Card        0.361128
Savings Account    0.295977
Money Transfer     0.188228
Personal Loan      0.154667
Name: proportion, dtype: float64

## Chunking

In [9]:
documents = split_documents(
    sample_df,
    chunk_size=500,
    chunk_overlap=100
)

## Display number of chunks

In [10]:
len(documents)

36817

## Display one chunk

In [11]:
documents[0]

Document(metadata={'complaint_id': 12543891, 'product': 'Credit Card'}, page_content='case numbers xxxx xxxx xxxx xxxx i spoke with cred ai representatives including supervisor xxxx and agents xxxx and xxxx regarding the closure of my account and the retention of my personal information i informed them that retaining my information despite no active business engagement constitutes identity theft they responded that my information could only be archived which effectively forces me to engage in business with them additionally my application was denied due to the reason unable to')

## Embedding model

In [12]:
embedding_model = create_embeddings()

## Generate embeddings

In [13]:
vector_store = build_faiss_index(
    documents,
    embedding_model
)
print(vector_store.index.ntotal)

36817


## Save vector database

In [14]:
vector_store.save_local("../vector_store")

## Reload test

In [15]:
db = FAISS.load_local(
    "../vector_store",
    embedding_model,
    allow_dangerous_deserialization=True
)

print(db.index.ntotal)

36817


## Retrieval test

In [16]:
results = db.similarity_search(
    "credit card charged twice",
    k=3
)

In [17]:
for r in results:
    print("="*80)
    print(r.page_content[:500])
    print(r.metadata)

was there again and i complained i did not want another new card as that doesn't seem to help this month the charge appeared twice when i called i was told that they would open a fraud investigation and would send another card i asked that they credit my account not to send a new card and to cancel the one i have i don't understand how the same charge can appear on 3 different cards if they had truly stopped the charge and investigated fraud
{'complaint_id': 10265366, 'product': 'Credit Card'}
i was recently reviewing my credit card statements from my financial institution and i noticed that i was incurring interest fees for a charge that was refunded as it was unauthorized which i had spoken with the merchant to resolve prior to even reaching out to my bank now recently i just noticed that i was charged twice for interest fees when there wasnt a valid balance in the first place the first interest charge was on xx xx xxxx in the amount of 47 00 and the second interest charge was on xx
